# Stock Market Analysis Agent

A **LangGraph**-based agent that fetches 60 days of historical stock data, computes
SMA 10, SMA 20 and RSI 14, then generates a **BUY / HOLD / SELL** recommendation.

## Agent Architecture

```
START
  |-> validate_ticker
        |-> fetch_stock_data       (yfinance -- 60 days)
              |-> calculate_indicators  (SMA10, SMA20, RSI14)
                    |-> generate_recommendation  (BUY/HOLD/SELL)
                          |-> format_report
                                |-> END
```

## Recommendation Logic

| Condition | Signal |
|-----------|--------|
| SMA10 > SMA20 AND RSI < 70 | **BUY** (golden cross, not overbought) |
| SMA10 < SMA20 AND RSI > 30 | **SELL** (death cross, not oversold) |
| Otherwise | **HOLD** |

## Setup

Ensure dependencies are installed before running:

In [1]:
%pip install langgraph yfinance pandas --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys, os
# Make the stock_agent package importable from this notebook
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

from stock_agent.graph import stock_agent
from stock_agent.state import StockAgentState

print('Agent loaded successfully.')
print(f'Nodes: {list(stock_agent.get_graph().nodes.keys())}')

Agent loaded successfully.
Nodes: ['__start__', 'validate_ticker', 'fetch_stock_data', 'calculate_indicators', 'generate_recommendation', 'format_report', '__end__']


## Helper: run the agent for any ticker

In [3]:
def analyse(ticker: str) -> None:
    """Run the stock agent and print the report."""
    state = StockAgentState(
        ticker=ticker,
        raw_data=None,
        indicators=None,
        recommendation=None,
        report=None,
        error=None,
    )
    final = stock_agent.invoke(state)
    print(final['report'])

---
## Demo 1 — Apple Inc. (AAPL)

In [4]:
analyse('AAPL')

  STOCK MARKET ANALYSIS REPORT - AAPL
  Date        : 2026-06-10
  Data Range  : 2026-03-23 to 2026-06-09
  Trading Days: 55
------------------------------------------------------------
  PRICE SUMMARY
  Current Price : $290.55
  60-Day Change : +15.64%
------------------------------------------------------------
  TECHNICAL INDICATORS
  SMA (10-day)  : $307.79
  SMA (20-day)  : $304.56
  RSI (14-day)  : 42.5
------------------------------------------------------------
  SIGNAL INTERPRETATION
  SMA Cross     : Bullish (SMA10 > SMA20)
  RSI Status    : Neutral (30-70)
------------------------------------------------------------
  RECOMMENDATION: BUY
  Reason        : Golden cross (SMA10 > SMA20) with RSI not overbought


## Demo 2 — Microsoft (MSFT)

In [5]:
analyse('MSFT')

  STOCK MARKET ANALYSIS REPORT - MSFT
  Date        : 2026-06-10
  Data Range  : 2026-03-23 to 2026-06-09
  Trading Days: 55
------------------------------------------------------------
  PRICE SUMMARY
  Current Price : $403.41
  60-Day Change : +5.56%
------------------------------------------------------------
  TECHNICAL INDICATORS
  SMA (10-day)  : $427.89
  SMA (20-day)  : $421.63
  RSI (14-day)  : 44.4
------------------------------------------------------------
  SIGNAL INTERPRETATION
  SMA Cross     : Bullish (SMA10 > SMA20)
  RSI Status    : Neutral (30-70)
------------------------------------------------------------
  RECOMMENDATION: BUY
  Reason        : Golden cross (SMA10 > SMA20) with RSI not overbought


## Demo 3 — Google / Alphabet (GOOGL)

In [6]:
analyse('GOOGL')

  STOCK MARKET ANALYSIS REPORT - GOOGL
  Date        : 2026-06-10
  Data Range  : 2026-03-23 to 2026-06-09
  Trading Days: 55
------------------------------------------------------------
  PRICE SUMMARY
  Current Price : $364.26
  60-Day Change : +20.66%
------------------------------------------------------------
  TECHNICAL INDICATORS
  SMA (10-day)  : $372.30
  SMA (20-day)  : $382.08
  RSI (14-day)  : 33.0
------------------------------------------------------------
  SIGNAL INTERPRETATION
  SMA Cross     : Bearish (SMA10 < SMA20)
  RSI Status    : Neutral (30-70)
------------------------------------------------------------
  RECOMMENDATION: SELL
  Reason        : Death cross (SMA10 < SMA20) with RSI not oversold


## Demo 4 — Tesla (TSLA)

In [7]:
analyse('TSLA')

  STOCK MARKET ANALYSIS REPORT - TSLA
  Date        : 2026-06-10
  Data Range  : 2026-03-23 to 2026-06-09
  Trading Days: 55
------------------------------------------------------------
  PRICE SUMMARY
  Current Price : $396.68
  60-Day Change : +4.16%
------------------------------------------------------------
  TECHNICAL INDICATORS
  SMA (10-day)  : $419.67
  SMA (20-day)  : $422.49
  RSI (14-day)  : 47.2
------------------------------------------------------------
  SIGNAL INTERPRETATION
  SMA Cross     : Bearish (SMA10 < SMA20)
  RSI Status    : Neutral (30-70)
------------------------------------------------------------
  RECOMMENDATION: SELL
  Reason        : Death cross (SMA10 < SMA20) with RSI not oversold


---
## Error Handling Tests

In [8]:
# Invalid ticker (numbers not allowed)
analyse('123XY')

ERROR: Invalid ticker symbol: '123XY'. Must be 1-5 letters.


In [9]:
# Non-existent ticker (valid format but not a real stock)
analyse('XYZAB')

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: XYZAB"}}}


$XYZAB: possibly delisted; no timezone found



1 Failed download:


['XYZAB']: possibly delisted; no timezone found


ERROR: No data found for ticker 'XYZAB'. It may be delisted or invalid.


---
## Unit Tests

In [10]:
from langgraph.graph.state import CompiledStateGraph

# Graph is compiled
assert isinstance(stock_agent, CompiledStateGraph), 'FAIL: agent not compiled'
print('PASS: stock_agent is a compiled CompiledStateGraph')

# All 5 nodes present
expected_nodes = {'validate_ticker', 'fetch_stock_data', 'calculate_indicators',
                  'generate_recommendation', 'format_report'}
actual_nodes = set(stock_agent.get_graph().nodes.keys()) - {'__start__', '__end__'}
assert actual_nodes == expected_nodes, f'FAIL: nodes mismatch -- {actual_nodes}'
print(f'PASS: all 5 nodes present -- {sorted(actual_nodes)}')

# Ticker validation rejects bad input
from stock_agent.nodes import validate_ticker
from stock_agent.state import StockAgentState
bad = validate_ticker(StockAgentState(ticker='123', raw_data=None, indicators=None,
                                       recommendation=None, report=None, error=None))
assert bad['error'] is not None, 'FAIL: bad ticker should set error'
print('PASS: validate_ticker correctly rejects numeric ticker')

# Ticker normalised to uppercase
good = validate_ticker(StockAgentState(ticker='aapl', raw_data=None, indicators=None,
                                        recommendation=None, report=None, error=None))
assert good['ticker'] == 'AAPL', 'FAIL: ticker should be uppercased'
print('PASS: validate_ticker uppercases the ticker symbol')

# Recommendation logic
from stock_agent.nodes import generate_recommendation

buy_state = StockAgentState(ticker='TEST', raw_data=None, error=None, report=None, recommendation=None,
    indicators={'sma10': 150.0, 'sma20': 140.0, 'rsi14': 55.0, 'current_price': 155.0, 'price_change_pct': 2.0})
assert generate_recommendation(buy_state)['recommendation'] == 'BUY', 'FAIL: should be BUY'
print('PASS: BUY when SMA10 > SMA20 and RSI < 70')

sell_state = StockAgentState(ticker='TEST', raw_data=None, error=None, report=None, recommendation=None,
    indicators={'sma10': 130.0, 'sma20': 145.0, 'rsi14': 55.0, 'current_price': 130.0, 'price_change_pct': -3.0})
assert generate_recommendation(sell_state)['recommendation'] == 'SELL', 'FAIL: should be SELL'
print('PASS: SELL when SMA10 < SMA20 and RSI > 30')

hold_state = StockAgentState(ticker='TEST', raw_data=None, error=None, report=None, recommendation=None,
    indicators={'sma10': 150.0, 'sma20': 140.0, 'rsi14': 75.0, 'current_price': 152.0, 'price_change_pct': 1.0})
assert generate_recommendation(hold_state)['recommendation'] == 'HOLD', 'FAIL: should be HOLD'
print('PASS: HOLD when SMA10 > SMA20 but RSI >= 70 (overbought)')

print('\nAll tests passed!')

PASS: stock_agent is a compiled CompiledStateGraph
PASS: all 5 nodes present -- ['calculate_indicators', 'fetch_stock_data', 'format_report', 'generate_recommendation', 'validate_ticker']
PASS: validate_ticker correctly rejects numeric ticker
PASS: validate_ticker uppercases the ticker symbol
PASS: BUY when SMA10 > SMA20 and RSI < 70
PASS: SELL when SMA10 < SMA20 and RSI > 30
PASS: HOLD when SMA10 > SMA20 but RSI >= 70 (overbought)

All tests passed!
